# 🖥️ Unidad 3: Generación de Código Intermedio
### Procesadores del Lenguaje II — 40 Ejercicios Resueltos

**Referencia bibliográfica:**  
Aho, A. V., Sethi, R., & Lam, M. S. (2011). *Compiladores*. Pearson Educación de México.  
[[Acceso en línea]](https://isergiobernalesgarcia.edu.pe/wp-content/uploads/2025/10/Compiladores-Alfred-V.-Aho-Monica-S.-Lam-Ravi-Sethi-Jeffrey-D.-Ullman.pdf)

---

## 📋 Estructura del cuaderno

| Sección | Tema | Ejercicios |
|---|---|---|
| A | Introducción a la Representación Intermedia | E01–E05 |
| B | Árboles Sintácticos y Grafos Acíclicos Dirigidos (GAD) | E06–E10 |
| C | Código de Tres Direcciones: Cuádruplos y Triplos | E11–E17 |
| D | Traducción de Expresiones Aritméticas | E18–E25 |
| E | Expresiones Booleanas y Cortocircuito | E26–E33 |
| F | Sentencias de Control y Retroceso (Backpatching) | E34–E40 |

> 🟢 **RESUELTO** — Todos los ejercicios incluyen solución completa y comentarios explicativos.

---

In [42]:
# ========================================================
# CONFIGURACIÓN INICIAL — Ejecuta esta celda primero
# ========================================================
from itertools import count
import copy

# Generador de temporales (t1, t2, t3, ...)
_temp_counter = count(1)
def nueva_temporal():
    return f't{next(_temp_counter)}'

# Generador de etiquetas (L1, L2, L3, ...)
_label_counter = count(1)
def nueva_etiqueta():
    return f'L{next(_label_counter)}'

# Lista de código generado
codigo_generado = []
def emitir(instruccion):
    codigo_generado.append(instruccion)
    return instruccion

def mostrar_codigo():
    for i, instr in enumerate(codigo_generado):
        print(f'  ({i:02d})  {instr}')

def reset():
    global _temp_counter, _label_counter, codigo_generado
    _temp_counter = count(1)
    _label_counter = count(1)
    codigo_generado = []

print('✅ Utilidades cargadas correctamente.')
print('   Funciones disponibles: nueva_temporal(), nueva_etiqueta(), emitir(), mostrar_codigo(), reset()')

✅ Utilidades cargadas correctamente.
   Funciones disponibles: nueva_temporal(), nueva_etiqueta(), emitir(), mostrar_codigo(), reset()


---
## 🔷 SECCIÓN A — Introducción a la Representación Intermedia
---

### 🟢 E01 — ¿Por qué usar representación intermedia?

**Enunciado:** Una empresa tiene 4 lenguajes fuente (Python, Java, C++, Rust) y quiere compilar para 3 arquitecturas (x86, ARM, RISC-V). Compara el número de traductores necesarios **con** y **sin** representación intermedia.

In [43]:
# ============================================================
# E01 — RESUELTO
# ============================================================

lenguajes = ['Python', 'Java', 'C++', 'Rust']
arquitecturas = ['x86', 'ARM', 'RISC-V']

m = len(lenguajes)       # 4 lenguajes fuente
n = len(arquitecturas)   # 3 arquitecturas destino

# Sin RI: un traductor directo para cada par (lenguaje, arquitectura)
sin_ri = m * n

# Con RI: un front-end por lenguaje + un back-end por arquitectura
con_ri = m + n

print("="*55)
print("  Ventaja de la Representación Intermedia (RI)")
print("="*55)
print(f"\n  Lenguajes fuente : {lenguajes}")
print(f"  Arquitecturas    : {arquitecturas}")
print(f"\n  Sin RI → m × n = {m} × {n} = {sin_ri} traductores")
print(f"  Con RI → m + n = {m} + {n} = {con_ri}  traductores")
print(f"\n  📉 Ahorro: {sin_ri - con_ri} módulos de traducción")
print("\n  💡 La RI separa el análisis del lenguaje fuente")
print("     de la generación de código máquina, mejorando")
print("     portabilidad y facilitando la optimización.")

  Ventaja de la Representación Intermedia (RI)

  Lenguajes fuente : ['Python', 'Java', 'C++', 'Rust']
  Arquitecturas    : ['x86', 'ARM', 'RISC-V']

  Sin RI → m × n = 4 × 3 = 12 traductores
  Con RI → m + n = 4 + 3 = 7  traductores

  📉 Ahorro: 5 módulos de traducción

  💡 La RI separa el análisis del lenguaje fuente
     de la generación de código máquina, mejorando
     portabilidad y facilitando la optimización.


### 🟢 E02 — Propiedades de una buena RI

**Enunciado:** Escribe un programa que clasifique y muestre las propiedades clave que debe cumplir una buena representación intermedia según el libro de Aho et al.

In [44]:
# ============================================================
# E02 — RESUELTO
# ============================================================

propiedades_ri = [
    ("Independencia de la máquina",
     "No debe depender de detalles de ninguna arquitectura específica.",
     "Permite reutilizar el front-end para múltiples targets."),
    ("Cercanía al lenguaje máquina",
     "Debe ser traducible fácilmente a código objeto.",
     "Simplifica la fase de generación de código final."),
    ("Facilidad de optimización",
     "Debe permitir aplicar transformaciones de mejora de rendimiento.",
     "Eliminación de código muerto, propagación de constantes."),
    ("Expresividad",
     "Debe poder representar todas las construcciones del lenguaje fuente.",
     "Asignaciones, saltos, llamadas a función, arreglos, etc."),
    ("Simplicidad",
     "Las instrucciones deben ser simples (máx. 3 operandos).",
     "Facilita el análisis y la manipulación algorítmica.")
]

print("  PROPIEDADES DE UNA BUENA REPRESENTACIÓN INTERMEDIA")
print("="*60)
for i, (nombre, descripcion, ejemplo) in enumerate(propiedades_ri, 1):
    print(f"\n  {i}. {nombre}")
    print(f"     Descripción : {descripcion}")
    print(f"     Ejemplo/Uso : {ejemplo}")

  PROPIEDADES DE UNA BUENA REPRESENTACIÓN INTERMEDIA

  1. Independencia de la máquina
     Descripción : No debe depender de detalles de ninguna arquitectura específica.
     Ejemplo/Uso : Permite reutilizar el front-end para múltiples targets.

  2. Cercanía al lenguaje máquina
     Descripción : Debe ser traducible fácilmente a código objeto.
     Ejemplo/Uso : Simplifica la fase de generación de código final.

  3. Facilidad de optimización
     Descripción : Debe permitir aplicar transformaciones de mejora de rendimiento.
     Ejemplo/Uso : Eliminación de código muerto, propagación de constantes.

  4. Expresividad
     Descripción : Debe poder representar todas las construcciones del lenguaje fuente.
     Ejemplo/Uso : Asignaciones, saltos, llamadas a función, arreglos, etc.

  5. Simplicidad
     Descripción : Las instrucciones deben ser simples (máx. 3 operandos).
     Ejemplo/Uso : Facilita el análisis y la manipulación algorítmica.


### 🟢 E03 — Posición de la RI en el compilador

**Enunciado:** Escribe una función `etapa_compilador(fase)` que, dado el nombre de una fase del compilador, retorne si pertenece al **front-end**, al **back-end** o es la propia **RI**. Muestra el pipeline completo.

In [45]:
# ============================================================
# E03 — RESUELTO
# ============================================================

def etapa_compilador(fase):
    """Clasifica cada fase del compilador en su posición del pipeline."""
    clasificacion = {
        'lexico'            : 'Front-end  (Análisis)',
        'sintactico'        : 'Front-end  (Análisis)',
        'semantico'         : 'Front-end  (Análisis)',
        'codigo_intermedio' : '──▶ Representación Intermedia ◀──',
        'optimizacion'      : 'Back-end   (Síntesis)',
        'generacion_codigo' : 'Back-end   (Síntesis)',
    }
    return clasificacion.get(fase, 'Fase desconocida')

fases = [
    'lexico', 'sintactico', 'semantico',
    'codigo_intermedio',
    'optimizacion', 'generacion_codigo'
]

print("  PIPELINE DEL COMPILADOR")
print("="*55)
for i, fase in enumerate(fases, 1):
    resultado = etapa_compilador(fase)
    print(f"  {i}. {fase:25s} → {resultado}")

print("\n  💡 La RI es el punto de transición entre el análisis")
print("     del lenguaje fuente y la síntesis del código objeto.")

  PIPELINE DEL COMPILADOR
  1. lexico                    → Front-end  (Análisis)
  2. sintactico                → Front-end  (Análisis)
  3. semantico                 → Front-end  (Análisis)
  4. codigo_intermedio         → ──▶ Representación Intermedia ◀──
  5. optimizacion              → Back-end   (Síntesis)
  6. generacion_codigo         → Back-end   (Síntesis)

  💡 La RI es el punto de transición entre el análisis
     del lenguaje fuente y la síntesis del código objeto.


### 🟢 E04 — Tipos de representación intermedia

**Enunciado:** Crea una clase `TipoRI` que represente los principales tipos de representación intermedia. Instancia al menos 4 tipos y muéstralos ordenados por nivel de abstracción.

In [46]:
# ============================================================
# E04 — RESUELTO
# ============================================================

class TipoRI:
    def __init__(self, nombre, descripcion, ventaja, nivel_abstraccion):
        self.nombre = nombre
        self.descripcion = descripcion
        self.ventaja = ventaja
        self.nivel_abstraccion = nivel_abstraccion  # 1=bajo, 5=alto

    def __repr__(self):
        return (f"  [Nivel {self.nivel_abstraccion}] {self.nombre}\n"
                f"    Descripción: {self.descripcion}\n"
                f"    Ventaja    : {self.ventaja}")

tipos_ri = [
    TipoRI(
        nombre="Código de Tres Direcciones",
        descripcion="Instrucciones con máx. 3 operandos, similar a ensamblador abstracto.",
        ventaja="Fácil de optimizar y generar código máquina.",
        nivel_abstraccion=2
    ),
    TipoRI(
        nombre="AST (Árbol de Sintaxis Abstracta)",
        descripcion="Árbol que refleja la estructura sintáctica del programa.",
        ventaja="Preserva la estructura jerárquica para análisis semántico.",
        nivel_abstraccion=4
    ),
    TipoRI(
        nombre="GAD (Grafo Acíclico Dirigido)",
        descripcion="Variante del AST que comparte nodos para subexpresiones comunes.",
        ventaja="Representación compacta, detecta redundancias automáticamente.",
        nivel_abstraccion=3
    ),
    TipoRI(
        nombre="Código P (Máquina de pila)",
        descripcion="Instrucciones para una máquina de pila abstracta (ej. JVM bytecode).",
        ventaja="Muy portable, simple de generar para expresiones.",
        nivel_abstraccion=2
    ),
    TipoRI(
        nombre="SSA (Asignación Estática Única)",
        descripcion="Forma especial de código de 3 direcciones donde cada var se asigna una sola vez.",
        ventaja="Facilita optimizaciones avanzadas como propagación de constantes.",
        nivel_abstraccion=2
    ),
]

tipos_ordenados = sorted(tipos_ri, key=lambda x: x.nivel_abstraccion, reverse=True)

print("  TIPOS DE RI — Ordenados de mayor a menor abstracción")
print("="*65)
for t in tipos_ordenados:
    print(t)
    print()

  TIPOS DE RI — Ordenados de mayor a menor abstracción
  [Nivel 4] AST (Árbol de Sintaxis Abstracta)
    Descripción: Árbol que refleja la estructura sintáctica del programa.
    Ventaja    : Preserva la estructura jerárquica para análisis semántico.

  [Nivel 3] GAD (Grafo Acíclico Dirigido)
    Descripción: Variante del AST que comparte nodos para subexpresiones comunes.
    Ventaja    : Representación compacta, detecta redundancias automáticamente.

  [Nivel 2] Código de Tres Direcciones
    Descripción: Instrucciones con máx. 3 operandos, similar a ensamblador abstracto.
    Ventaja    : Fácil de optimizar y generar código máquina.

  [Nivel 2] Código P (Máquina de pila)
    Descripción: Instrucciones para una máquina de pila abstracta (ej. JVM bytecode).
    Ventaja    : Muy portable, simple de generar para expresiones.

  [Nivel 2] SSA (Asignación Estática Única)
    Descripción: Forma especial de código de 3 direcciones donde cada var se asigna una sola vez.
    Ventaja    : Fac

### 🟢 E05 — Ventajas de la RI con múltiples lenguajes

**Enunciado:** Dado un número `m` de lenguajes fuente y `n` de arquitecturas destino, calcula la **reducción porcentual** en el número de traductores al usar RI. Pruébala con varios valores.

In [47]:
# ============================================================
# E05 — RESUELTO
# ============================================================

def reduccion_ri(m, n):
    """
    Calcula la reducción al usar RI.
    Sin RI: m*n traductores. Con RI: m+n traductores.
    Retorna (sin_ri, con_ri, reduccion_porcentual).
    """
    sin_ri = m * n
    con_ri = m + n
    reduccion = ((sin_ri - con_ri) / sin_ri) * 100
    return sin_ri, con_ri, reduccion

casos = [(2, 2), (5, 5), (10, 10), (20, 20), (10, 3), (3, 10)]

print(f"  {'m':>4} {'n':>4} | {'Sin RI':>8} {'Con RI':>8} {'Reducción':>12}")
print("  " + "-"*48)
for m, n in casos:
    sin_ri, con_ri, reduccion = reduccion_ri(m, n)
    print(f"  {m:>4} {n:>4} | {sin_ri:>8} {con_ri:>8} {reduccion:>10.1f}%")

print("\n  💡 Cuanto mayor es m×n, mayor es el ahorro porcentual.")
print("     El caso (20,20) sin RI: 400 traductores, con RI: solo 40.")

     m    n |   Sin RI   Con RI    Reducción
  ------------------------------------------------
     2    2 |        4        4        0.0%
     5    5 |       25       10       60.0%
    10   10 |      100       20       80.0%
    20   20 |      400       40       90.0%
    10    3 |       30       13       56.7%
     3   10 |       30       13       56.7%

  💡 Cuanto mayor es m×n, mayor es el ahorro porcentual.
     El caso (20,20) sin RI: 400 traductores, con RI: solo 40.


---
## 🔷 SECCIÓN B — Árboles Sintácticos y Grafos Acíclicos Dirigidos (GAD)
---

### 🟢 E06 — Construcción de un AST para expresión aritmética

**Enunciado:** Construye el AST para la expresión `a + b * c` y muéstralo.

In [48]:
# ============================================================
# E06 — RESUELTO
# ============================================================

class NodoAST:
    def __init__(self, valor, izq=None, der=None):
        self.valor = valor
        self.izq = izq
        self.der = der

    def mostrar(self, nivel=0, prefijo='Raíz: '):
        print(" " * (nivel * 4) + prefijo + str(self.valor))
        if self.izq:
            self.izq.mostrar(nivel + 1, 'Izq:  ')
        if self.der:
            self.der.mostrar(nivel + 1, 'Der:  ')

# Expresión: a + b * c
# La precedencia hace que b * c se evalúe primero
nodo_a    = NodoAST('a')
nodo_b    = NodoAST('b')
nodo_c    = NodoAST('c')
nodo_mult = NodoAST('*', nodo_b, nodo_c)    # b * c
nodo_suma = NodoAST('+', nodo_a, nodo_mult) # a + (b*c)

print("  AST para la expresión: a + b * c")
print("  (La precedencia queda reflejada en la estructura)")
print()
nodo_suma.mostrar()
print()
print("  💡 La multiplicación es el subárbol derecho de la suma,")
print("     garantizando que se evalúa primero.")

  AST para la expresión: a + b * c
  (La precedencia queda reflejada en la estructura)

Raíz: +
    Izq:  a
    Der:  *
        Izq:  b
        Der:  c

  💡 La multiplicación es el subárbol derecho de la suma,
     garantizando que se evalúa primero.


### 🟢 E07 — Construcción de un GAD (Grafo Acíclico Dirigido)

**Enunciado:** Construye el GAD para `a + a * (b - c) + (b - c) * d` usando numeración de valores. Identifica las subexpresiones comunes.

In [49]:
# ============================================================
# E07 — RESUELTO
# ============================================================

class ConstructorGAD:
    """Construye un GAD usando numeración de valores (tabla hash)."""

    def __init__(self):
        self.tabla  = {}   # firma -> id_nodo
        self.nodos  = {}   # id_nodo -> descripcion
        self._id    = 0
        self.reusos = []

    def _nuevo_id(self):
        self._id += 1
        return self._id

    def hoja(self, nombre):
        firma = ('hoja', nombre)
        if firma in self.tabla:
            self.reusos.append(f'  ✅ REUTILIZADA: Hoja({nombre}) → Nodo {self.tabla[firma]}')
            return self.tabla[firma]
        nid = self._nuevo_id()
        self.tabla[firma] = nid
        self.nodos[nid] = f'Hoja({nombre})'
        return nid

    def interior(self, op, v1, v2):
        firma = (op, v1, v2)
        if firma in self.tabla:
            nid = self.tabla[firma]
            self.reusos.append(f'  ✅ REUTILIZADA: ({op}, n{v1}, n{v2}) → Nodo {nid} [{self.nodos[nid]}]')
            return nid
        nid = self._nuevo_id()
        self.tabla[firma] = nid
        self.nodos[nid] = f'Interior({op}, n{v1}, n{v2})'
        return nid

gad = ConstructorGAD()

# Hojas
na = gad.hoja('a')
nb = gad.hoja('b')
nc = gad.hoja('c')
nd = gad.hoja('d')

# b - c (primera aparición)
n_bc = gad.interior('-', nb, nc)
# a * (b - c)
n_a_mult_bc = gad.interior('*', na, n_bc)
# a + a*(b-c)
n_suma1 = gad.interior('+', na, n_a_mult_bc)
# (b - c) * d  — b-c se reutiliza
n_bc_mult_d = gad.interior('*', n_bc, nd)
# resultado final
n_final = gad.interior('+', n_suma1, n_bc_mult_d)

print("  GAD para: a + a * (b - c) + (b - c) * d")
print("="*55)
print("\n  Nodos creados:")
for nid, desc in gad.nodos.items():
    print(f"    Nodo {nid}: {desc}")
print("\n  Subexpresiones reutilizadas (ahorro):")
for r in gad.reusos:
    print(r)
print(f"\n  Nodo raíz: {n_final}  |  Total nodos GAD: {gad._id}")

  GAD para: a + a * (b - c) + (b - c) * d

  Nodos creados:
    Nodo 1: Hoja(a)
    Nodo 2: Hoja(b)
    Nodo 3: Hoja(c)
    Nodo 4: Hoja(d)
    Nodo 5: Interior(-, n2, n3)
    Nodo 6: Interior(*, n1, n5)
    Nodo 7: Interior(+, n1, n6)
    Nodo 8: Interior(*, n5, n4)
    Nodo 9: Interior(+, n7, n8)

  Subexpresiones reutilizadas (ahorro):

  Nodo raíz: 9  |  Total nodos GAD: 9


### 🟢 E08 — Recorrido en postorden del AST para generar código

**Enunciado:** Implementa un recorrido en postorden del AST que genere código de tres direcciones para `(a + b) * (a - b)`.

In [50]:
# ============================================================
# E08 — RESUELTO
# ============================================================
reset()

class NodoCodigo:
    def __init__(self, valor, izq=None, der=None):
        self.valor = valor
        self.izq   = izq
        self.der   = der

def generar_codigo_expr(nodo):
    """Recorrido postorden: genera código y retorna el lugar del resultado."""
    if nodo.izq is None and nodo.der is None:
        return nodo.valor  # hoja: el lugar es el propio nombre

    lugar_izq = generar_codigo_expr(nodo.izq)
    lugar_der = generar_codigo_expr(nodo.der)

    temp = nueva_temporal()
    emitir(f'{temp} := {lugar_izq} {nodo.valor} {lugar_der}')
    return temp

# AST para (a + b) * (a - b)
ast = NodoCodigo('*',
    NodoCodigo('+', NodoCodigo('a'), NodoCodigo('b')),
    NodoCodigo('-', NodoCodigo('a'), NodoCodigo('b'))
)

resultado = generar_codigo_expr(ast)

print("  Expresión: (a + b) * (a - b)")
print("\n  Código de tres direcciones generado:")
mostrar_codigo()
print(f"\n  Resultado final en: {resultado}")
print("\n  💡 El postorden garantiza que los operandos se calculan")
print("     ANTES que la operación que los necesita.")

  Expresión: (a + b) * (a - b)

  Código de tres direcciones generado:
  (00)  t1 := a + b
  (01)  t2 := a - b
  (02)  t3 := t1 * t2

  Resultado final en: t3

  💡 El postorden garantiza que los operandos se calculan
     ANTES que la operación que los necesita.


### 🟢 E09 — Contar nodos de un AST vs GAD

**Enunciado:** Para la expresión `x * x + 2 * x + 1`, compara el número de nodos en el AST (sin compartir) y en el GAD (compartiendo subexpresiones comunes).

In [51]:
# ============================================================
# E09 — RESUELTO
# ============================================================

# --- AST: cada ocurrencia de una variable crea un nodo propio ---
# Expresión: x * x + 2 * x + 1
# Árbol:
#          +
#         / \
#        +   1
#       / \
#     x*x  2*x
#     /\    /\
#    x  x  2  x
#
# Nodos del AST:
#   Operadores : +, +, *, *  → 4 nodos
#   Hojas      : x, x, 2, x, 1  → 5 nodos
#   Total      : 9 nodos

nodos_ast = 9  # conteo manual

# --- GAD: las subexpresiones comunes se comparten ---
gad2 = ConstructorGAD()

# Hojas únicas
nx = gad2.hoja('x')   # solo 1 nodo para x
n2 = gad2.hoja('2')
n1 = gad2.hoja('1')

# x * x
n_xx    = gad2.interior('*', nx, nx)
# 2 * x
n_2x    = gad2.interior('*', n2, nx)
# x*x + 2*x
n_sum1  = gad2.interior('+', n_xx, n_2x)
# (x*x + 2*x) + 1
n_final = gad2.interior('+', n_sum1, n1)

nodos_gad = gad2._id

print("  Expresión: x * x + 2 * x + 1")
print("="*45)
print(f"  Nodos en el AST : {nodos_ast}  (x aparece 3 veces como nodos distintos)")
print(f"  Nodos en el GAD : {nodos_gad}  (x es 1 solo nodo compartido)")
print(f"  Ahorro          : {nodos_ast - nodos_gad} nodos")
print()
print("  Nodos del GAD:")
for nid, desc in gad2.nodos.items():
    print(f"    Nodo {nid}: {desc}")

  Expresión: x * x + 2 * x + 1
  Nodos en el AST : 9  (x aparece 3 veces como nodos distintos)
  Nodos en el GAD : 7  (x es 1 solo nodo compartido)
  Ahorro          : 2 nodos

  Nodos del GAD:
    Nodo 1: Hoja(x)
    Nodo 2: Hoja(2)
    Nodo 3: Hoja(1)
    Nodo 4: Interior(*, n1, n1)
    Nodo 5: Interior(*, n2, n1)
    Nodo 6: Interior(+, n4, n5)
    Nodo 7: Interior(+, n6, n3)


### 🟢 E10 — AST para sentencia if-else

**Enunciado:** Modela el AST para `if (x > 0) y = x; else y = -x;` y dibuja el árbol resultante.

In [52]:
# ============================================================
# E10 — RESUELTO
# ============================================================

class NodoAST3:
    """Nodo AST con soporte para hasta 3 hijos."""
    def __init__(self, valor, hijo1=None, hijo2=None, hijo3=None):
        self.valor = valor
        self.hijos = [h for h in [hijo1, hijo2, hijo3] if h is not None]

    def mostrar(self, nivel=0, prefijo='Raíz: '):
        print(' ' * (nivel * 4) + prefijo + str(self.valor))
        etiquetas = ['Cond:  ', 'Then:  ', 'Else:  ']
        for etiqueta, hijo in zip(etiquetas, self.hijos):
            hijo.mostrar(nivel + 1, etiqueta)

# Condición: x > 0
condicion = NodoAST3('>', NodoAST3('x'), NodoAST3('0'))

# Rama then: y = x
rama_then = NodoAST3('asignar', NodoAST3('y'), NodoAST3('x'))

# Menos unario para -x
neg_x     = NodoAST3('uminus', NodoAST3('x'))
# Rama else: y = -x
rama_else = NodoAST3('asignar', NodoAST3('y'), neg_x)

# Nodo raíz if-else con 3 hijos
ast_if = NodoAST3('if-else', condicion, rama_then, rama_else)

print("  AST para: if (x > 0) y = x; else y = -x;")
print()
ast_if.mostrar()
print()
print("  💡 El nodo if-else tiene 3 hijos: condición, rama-then, rama-else.")
print("     La estructura arborescente hace explícito el flujo de control.")

  AST para: if (x > 0) y = x; else y = -x;

Raíz: if-else
    Cond:  >
        Cond:  x
        Then:  0
    Then:  asignar
        Cond:  y
        Then:  x
    Else:  asignar
        Cond:  y
        Then:  uminus
            Cond:  x

  💡 El nodo if-else tiene 3 hijos: condición, rama-then, rama-else.
     La estructura arborescente hace explícito el flujo de control.


---
## 🔷 SECCIÓN C — Código de Tres Direcciones: Cuádruplos y Triplos
---

### 🟢 E11 — Generar cuádruplos para una expresión

**Enunciado:** Genera los cuádruplos para `a = b * -c + b * -c` en formato `(op, arg1, arg2, resultado)`.

In [53]:
# ============================================================
# E11 — RESUELTO
# ============================================================

# Cuádruplo: (operador, arg1, arg2, resultado)
# '' significa campo vacío
cuadruplos = [
    ('uminus', 'c',  '',   't1'),  # t1 = -c
    ('*',      'b',  't1', 't2'),  # t2 = b * t1
    ('uminus', 'c',  '',   't3'),  # t3 = -c  (segunda aparición)
    ('*',      'b',  't3', 't4'),  # t4 = b * t3
    ('+',      't2', 't4', 't5'),  # t5 = t2 + t4
    (':=',     't5', '',   'a'),   # a  = t5
]

print("  Sentencia: a = b * -c + b * -c")
print("\n  Cuádruplos generados:")
print(f"  {'Pos':>4} {'Operador':>10} {'Arg1':>6} {'Arg2':>6} {'Resultado':>10}")
print("  " + "-"*42)
for i, (op, a1, a2, res) in enumerate(cuadruplos):
    print(f"  ({i:02d}) {op:>10} {a1:>6} {a2:>6} {res:>10}")

print("\n  💡 Nota: t3 = t1 = -c. Un GAD unificaría ambas en un solo nodo.")

  Sentencia: a = b * -c + b * -c

  Cuádruplos generados:
   Pos   Operador   Arg1   Arg2  Resultado
  ------------------------------------------
  (00)     uminus      c                t1
  (01)          *      b     t1         t2
  (02)     uminus      c                t3
  (03)          *      b     t3         t4
  (04)          +     t2     t4         t5
  (05)         :=     t5                 a

  💡 Nota: t3 = t1 = -c. Un GAD unificaría ambas en un solo nodo.


### 🟢 E12 — Comparar cuádruplos vs triplos

**Enunciado:** Para `a = b * -c + b * -c`, genera la representación en **triplos** y compárala con los cuádruplos.

In [54]:
# ============================================================
# E12 — RESUELTO
# ============================================================

# En triplos los resultados se referencian por posición (n)
triplos = [
    ('uminus', 'c',   ''),    # (0): -c
    ('*',      'b',   '(0)'), # (1): b * (0)  →  b * (-c)
    ('uminus', 'c',   ''),    # (2): -c  (segunda vez)
    ('*',      'b',   '(2)'), # (3): b * (2)
    ('+',      '(1)', '(3)'), # (4): (1) + (3)
    (':=',     'a',   '(4)'), # (5): a := (4)
]

print("  TRIPLOS para: a = b * -c + b * -c")
print(f"\n  {'Pos':>4} {'Operador':>10} {'Arg1':>6} {'Arg2':>6}")
print("  " + "-"*35)
for i, (op, a1, a2) in enumerate(triplos):
    print(f"  ({i}) {op:>10} {a1:>6} {a2:>6}")

print("\n  COMPARACIÓN cuádruplos vs triplos:")
print("  ├─ Cuádruplos : 4 campos, usa nombres de temporales (t1, t2, ...)")
print("  └─ Triplos    : 3 campos, referencias posicionales ((0), (1), ...)")
print("\n  ⚠️  Mover instrucciones en triplos exige actualizar TODAS")
print("      las referencias posicionales → optimización más difícil.")

  TRIPLOS para: a = b * -c + b * -c

   Pos   Operador   Arg1   Arg2
  -----------------------------------
  (0)     uminus      c       
  (1)          *      b    (0)
  (2)     uminus      c       
  (3)          *      b    (2)
  (4)          +    (1)    (3)
  (5)         :=      a    (4)

  COMPARACIÓN cuádruplos vs triplos:
  ├─ Cuádruplos : 4 campos, usa nombres de temporales (t1, t2, ...)
  └─ Triplos    : 3 campos, referencias posicionales ((0), (1), ...)

  ⚠️  Mover instrucciones en triplos exige actualizar TODAS
      las referencias posicionales → optimización más difícil.


### 🟢 E13 — Triplos indirectos

**Enunciado:** Implementa una simulación de **triplos indirectos** para demostrar que reordenar instrucciones es posible sin modificar el arreglo de triplos.

In [55]:
# ============================================================
# E13 — RESUELTO
# ============================================================

arreglo_triplos = [
    (0, 'uminus', 'c',   ''),
    (1, '*',      'b',   '(0)'),
    (2, 'uminus', 'c',   ''),    # duplicado de (0)
    (3, '*',      'b',   '(2)'),
    (4, '+',      '(1)', '(3)'),
    (5, ':=',     'a',   '(4)'),
]

# Orden original
orden_original  = [0, 1, 2, 3, 4, 5]
# Orden optimizado: se elimina (2) reutilizando (0)
orden_optimizado = [0, 1, 3, 4, 5]  # triplo (2) eliminado

def ejecutar_plan(orden, triplos):
    print(f"  {'Paso':>5} → Instrucción")
    print("  " + "-"*40)
    for paso, idx in enumerate(orden):
        t = triplos[idx]
        print(f"  Paso {paso+1}  → Triplo({t[0]}): {t[1]:>8}  {t[2]:>5}  {t[3]:>5}")

print("  TRIPLOS INDIRECTOS — El arreglo es inmutable.")
print("  La lista de ejecución se reordena sin tocar los triplos.\n")

print("  Arreglo de triplos (fijo):")
for t in arreglo_triplos:
    print(f"    ({t[0]}): {t[1]:>8}  {t[2]:>5}  {t[3]:>5}")

print("\n  EJECUCIÓN ORIGINAL:", orden_original)
ejecutar_plan(orden_original, arreglo_triplos)

print("\n  EJECUCIÓN OPTIMIZADA:", orden_optimizado)
print("  (triplo 2 eliminado — se reutiliza el resultado del triplo 0)")
ejecutar_plan(orden_optimizado, arreglo_triplos)

  TRIPLOS INDIRECTOS — El arreglo es inmutable.
  La lista de ejecución se reordena sin tocar los triplos.

  Arreglo de triplos (fijo):
    (0):   uminus      c       
    (1):        *      b    (0)
    (2):   uminus      c       
    (3):        *      b    (2)
    (4):        +    (1)    (3)
    (5):       :=      a    (4)

  EJECUCIÓN ORIGINAL: [0, 1, 2, 3, 4, 5]
   Paso → Instrucción
  ----------------------------------------
  Paso 1  → Triplo(0):   uminus      c       
  Paso 2  → Triplo(1):        *      b    (0)
  Paso 3  → Triplo(2):   uminus      c       
  Paso 4  → Triplo(3):        *      b    (2)
  Paso 5  → Triplo(4):        +    (1)    (3)
  Paso 6  → Triplo(5):       :=      a    (4)

  EJECUCIÓN OPTIMIZADA: [0, 1, 3, 4, 5]
  (triplo 2 eliminado — se reutiliza el resultado del triplo 0)
   Paso → Instrucción
  ----------------------------------------
  Paso 1  → Triplo(0):   uminus      c       
  Paso 2  → Triplo(1):        *      b    (0)
  Paso 3  → Triplo(3):    

### 🟢 E14 — Identificar tipos de instrucciones

**Enunciado:** Dada una lista de instrucciones de código de tres direcciones, clasifica cada una según su tipo.

In [56]:
# ============================================================
# E14 — RESUELTO
# ============================================================

def clasificar_instruccion(instr):
    """Clasifica una instrucción de código de tres direcciones."""
    instr = instr.strip()

    if instr.startswith('goto '):
        return 'salto_incondicional'
    if instr.startswith('if '):
        return 'salto_condicional'
    if instr.startswith('param '):
        return 'parametro'
    if instr.startswith('call '):
        return 'llamada'

    # Es una asignación: analizar la parte derecha del ':='
    if ':=' in instr:
        _, rhs = instr.split(':=', 1)
        tokens = rhs.strip().split()

        if len(tokens) == 1:
            return 'copia'
        if len(tokens) == 2:
            return 'asignacion_unaria'   # op operando
        if len(tokens) == 3:
            return 'asignacion_binaria'  # operando op operando

    return 'desconocida'

instrucciones = [
    't1 := a + b',
    't2 := -t1',
    'x := y',
    'goto L1',
    'if a < b goto L2',
    'param x',
    'call f, 1',
    't3 := a * b',
    't4 := not flag',
]

print("  Clasificación de instrucciones:")
print(f"  {'Instrucción':30s} → {'Tipo'}")
print("  " + "-"*58)
for instr in instrucciones:
    tipo = clasificar_instruccion(instr)
    print(f"  {instr:30s} → {tipo}")

  Clasificación de instrucciones:
  Instrucción                    → Tipo
  ----------------------------------------------------------
  t1 := a + b                    → asignacion_binaria
  t2 := -t1                      → copia
  x := y                         → copia
  goto L1                        → salto_incondicional
  if a < b goto L2               → salto_condicional
  param x                        → parametro
  call f, 1                      → llamada
  t3 := a * b                    → asignacion_binaria
  t4 := not flag                 → asignacion_unaria


### 🟢 E15 — Generar cuádruplos para múltiples asignaciones

**Enunciado:** Genera los cuádruplos para:  
1. `x = a + b`  
2. `y = x * c`  
3. `z = x + y - c`

In [57]:
# ============================================================
# E15 — RESUELTO
# ============================================================

cuadruplos = [
    # 1) x = a + b
    ('+',  'a', 'b',  'x'),   # x := a + b

    # 2) y = x * c
    ('*',  'x', 'c',  'y'),   # y := x * c

    # 3) z = x + y - c   →  t1 = x + y;  z = t1 - c
    ('+',  'x', 'y',  't1'),  # t1 := x + y
    ('-',  't1','c',  'z'),   # z  := t1 - c
]

print("  Sentencias:")
print("    x = a + b")
print("    y = x * c")
print("    z = x + y - c")
print("\n  Cuádruplos generados:")
print(f"  {'#':>3} {'Op':>6} {'Arg1':>6} {'Arg2':>6} {'Res':>6}")
print("  " + "-"*34)
for i, q in enumerate(cuadruplos):
    print(f"  ({i:02d}) {q[0]:>6} {q[1]:>6} {q[2]:>6} {q[3]:>6}")

# Verificación con valores: a=2, b=3, c=4
# x = 2+3 = 5 | y = 5*4 = 20 | z = 5+20-4 = 21
print("\n  Verificación (a=2, b=3, c=4):")
print("    x = 2+3 = 5  |  y = 5*4 = 20  |  z = 5+20-4 = 21")

  Sentencias:
    x = a + b
    y = x * c
    z = x + y - c

  Cuádruplos generados:
    #     Op   Arg1   Arg2    Res
  ----------------------------------
  (00)      +      a      b      x
  (01)      *      x      c      y
  (02)      +      x      y     t1
  (03)      -     t1      c      z

  Verificación (a=2, b=3, c=4):
    x = 2+3 = 5  |  y = 5*4 = 20  |  z = 5+20-4 = 21


### 🟢 E16 — Simular la ejecución de cuádruplos

**Enunciado:** Escribe un intérprete que ejecute una lista de cuádruplos y muestre el valor de las variables al final. Pruébalo con `a=5, b=3, c=2`.

In [58]:
# ============================================================
# E16 — RESUELTO
# ============================================================

def ejecutar_cuadruplos(cuadruplos, memoria_inicial):
    """Intérprete básico de cuádruplos."""
    memoria = dict(memoria_inicial)

    operaciones = {
        '+':  lambda x, y: x + y,
        '-':  lambda x, y: x - y,
        '*':  lambda x, y: x * y,
        '/':  lambda x, y: x / y,
    }

    def valor(v):
        """Retorna el valor de v desde memoria o como literal numérico."""
        if v in memoria:
            return memoria[v]
        try:
            return float(v)
        except ValueError:
            return v

    for op, a1, a2, res in cuadruplos:
        if op == ':=':
            memoria[res] = valor(a1)
        elif op == 'uminus':
            memoria[res] = -valor(a1)
        elif op in operaciones:
            memoria[res] = operaciones[op](valor(a1), valor(a2))

    return memoria

# Cuádruplos del ejercicio E15
cuadruplos_e15 = [
    ('+',  'a', 'b',  'x'),
    ('*',  'x', 'c',  'y'),
    ('+',  'x', 'y',  't1'),
    ('-',  't1','c',  'z'),
]

memoria_inicial = {'a': 5, 'b': 3, 'c': 2}
resultado = ejecutar_cuadruplos(cuadruplos_e15, memoria_inicial)

print("  Intérprete de cuádruplos")
print(f"  Valores iniciales: a={memoria_inicial['a']}, b={memoria_inicial['b']}, c={memoria_inicial['c']}")
print("\n  Memoria final:")
for var in sorted(resultado):
    print(f"    {var} = {resultado[var]}")

# Verificación
a, b, c = 5, 3, 2
x_esp = a + b
y_esp = x_esp * c
z_esp = x_esp + y_esp - c
print(f"\n  Verificación manual: x={x_esp}, y={y_esp}, z={z_esp}")
ok = resultado.get('z') == z_esp
print(f"  {'✅ Correcto' if ok else '❌ Error'}")

  Intérprete de cuádruplos
  Valores iniciales: a=5, b=3, c=2

  Memoria final:
    a = 5
    b = 3
    c = 2
    t1 = 24
    x = 8
    y = 16
    z = 22

  Verificación manual: x=8, y=16, z=22
  ✅ Correcto


### 🟢 E17 — Diferencia entre cuádruplos y triplos (ventajas de optimización)

**Enunciado:** Demuestra por qué mover instrucciones en cuádruplos es simple, pero en triplos exige actualizar todas las referencias posicionales.

In [59]:
# ============================================================
# E17 — RESUELTO
# ============================================================

def mover_instruccion_cuadruplos(lista, de, a):
    """
    Mueve el cuádruplo en posición 'de' a la posición 'a'.
    En cuádruplos los nombres de temporales (t1, t2…) no dependen
    de la posición en el arreglo, por lo que no hay nada más que actualizar.
    """
    item = lista.pop(de)
    lista.insert(a, item)

def mover_instruccion_triplos(lista, de, a):
    """
    Mueve el triplo en posición 'de' a la posición 'a' y actualiza
    TODAS las referencias posicionales en el arreglo.
    """
    item = lista.pop(de)
    lista.insert(a, item)

    # Después de mover, necesitamos reparar las referencias.
    # Este remapeo es simplificado; en la práctica es más complejo.
    for i, (op, arg1, arg2) in enumerate(lista):
        nuevo_arg1 = f'({a})' if arg1 == f'({de})' else arg1
        nuevo_arg2 = f'({a})' if arg2 == f'({de})' else arg2
        lista[i] = (op, nuevo_arg1, nuevo_arg2)

# --- Cuádruplos ---
cuadruplos_ej = [
    ('+', 'a', 'b', 't1'),
    ('*', 't1', 'c', 't2'),
    ('-', 't2', 'd', 't3'),
]

cq = copy.deepcopy(cuadruplos_ej)
print("  CUÁDRUPLOS — antes de mover posición 0→2:")
for i, q in enumerate(cq): print(f"    ({i}): {q}")
mover_instruccion_cuadruplos(cq, 0, 2)
print("  CUÁDRUPLOS — después (las temporales t1,t2 siguen siendo válidas):")
for i, q in enumerate(cq): print(f"    ({i}): {q}")

# --- Triplos ---
triplos_ej = [
    ('+', 'a',   'b'),
    ('*', '(0)', 'c'),
    ('-', '(1)', 'd'),
]

ct = copy.deepcopy(triplos_ej)
print("\n  TRIPLOS — antes de mover posición 0→2:")
for i, t in enumerate(ct): print(f"    ({i}): {t}")
mover_instruccion_triplos(ct, 0, 2)
print("  TRIPLOS — después (referencias actualizadas):")
for i, t in enumerate(ct): print(f"    ({i}): {t}")
print("\n  💡 En triplos, mover UNA instrucción puede obligar a actualizar")
print("     TODAS las demás referencias → mayor costo de optimización.")

  CUÁDRUPLOS — antes de mover posición 0→2:
    (0): ('+', 'a', 'b', 't1')
    (1): ('*', 't1', 'c', 't2')
    (2): ('-', 't2', 'd', 't3')
  CUÁDRUPLOS — después (las temporales t1,t2 siguen siendo válidas):
    (0): ('*', 't1', 'c', 't2')
    (1): ('-', 't2', 'd', 't3')
    (2): ('+', 'a', 'b', 't1')

  TRIPLOS — antes de mover posición 0→2:
    (0): ('+', 'a', 'b')
    (1): ('*', '(0)', 'c')
    (2): ('-', '(1)', 'd')
  TRIPLOS — después (referencias actualizadas):
    (0): ('*', '(2)', 'c')
    (1): ('-', '(1)', 'd')
    (2): ('+', 'a', 'b')

  💡 En triplos, mover UNA instrucción puede obligar a actualizar
     TODAS las demás referencias → mayor costo de optimización.


---
## 🔷 SECCIÓN D — Traducción de Expresiones Aritméticas
---

### 🟢 E18 — Esquema de traducción S-atribuido para expresiones

**Enunciado:** Implementa el esquema S-atribuido para generar código de tres direcciones desde un árbol de expresiones.

In [60]:
# ============================================================
# E18 — RESUELTO
# ============================================================
reset()

class ExprNode:
    """Nodo para expresiones con atributos sintetizados 'lugar' y 'codigo'."""
    def __init__(self, op, izq=None, der=None):
        self.op     = op
        self.izq    = izq
        self.der    = der
        self.lugar  = None
        self.codigo = []

def traducir_expr(nodo):
    """
    Esquema S-atribuido (Aho et al.):
      E → id  : E.lugar = id.nombre, E.codigo = []
      E → E1 op E2 : crea temporal t, emite t := E1.lugar op E2.lugar
    """
    if nodo.izq is None and nodo.der is None:
        nodo.lugar  = nodo.op
        nodo.codigo = []
        return

    traducir_expr(nodo.izq)
    traducir_expr(nodo.der)

    temp = nueva_temporal()
    instruccion = f'{temp} := {nodo.izq.lugar} {nodo.op} {nodo.der.lugar}'

    nodo.lugar  = temp
    nodo.codigo = nodo.izq.codigo + nodo.der.codigo + [instruccion]

# Expresión: x + y * z
expr = ExprNode('+',
    ExprNode('x'),
    ExprNode('*', ExprNode('y'), ExprNode('z'))
)

traducir_expr(expr)

print("  Expresión: x + y * z")
print("\n  Código de tres direcciones (esquema S-atribuido):")
for instruccion in expr.codigo:
    print(f'    {instruccion}')
print(f"\n  Resultado en: {expr.lugar}")
print("\n  💡 Postorden garantiza que y*z se genera ANTES que x+(…).")

  Expresión: x + y * z

  Código de tres direcciones (esquema S-atribuido):
    t1 := y * z
    t2 := x + t1

  Resultado en: t2

  💡 Postorden garantiza que y*z se genera ANTES que x+(…).


### 🟢 E19 — Traducción con conversión de tipos implícita

**Enunciado:** Implementa la generación de código para expresiones con tipos mezclados (int y float). Inserta `inttofloat` cuando sea necesario.

In [61]:
# ============================================================
# E19 — RESUELTO
# ============================================================
reset()

tabla_simbolos = {'i': 'int', 'j': 'int', 'x': 'float', 'y': 'float'}

def tipo(var):
    if '.' in str(var): return 'float'
    return tabla_simbolos.get(var, 'int')

def traducir_con_tipos(op, lugar1, tipo1, lugar2, tipo2):
    instrucciones = []

    if tipo1 == 'int' and tipo2 == 'float':
        t_conv = nueva_temporal()
        instrucciones.append(f'{t_conv} := inttofloat {lugar1}')
        lugar1  = t_conv
        tipo_res = 'float'
    elif tipo1 == 'float' and tipo2 == 'int':
        t_conv = nueva_temporal()
        instrucciones.append(f'{t_conv} := inttofloat {lugar2}')
        lugar2  = t_conv
        tipo_res = 'float'
    else:
        tipo_res = tipo1

    t_res = nueva_temporal()
    instrucciones.append(f'{t_res} := {lugar1} {op} {lugar2}')
    return t_res, tipo_res, instrucciones

casos = [
    ('i', 'x', '+'),  # int + float
    ('x', 'y', '*'),  # float + float
    ('x', 'i', '-'),  # float - int
    ('i', 'j', '+'),  # int + int
]

for l1, l2, op in casos:
    t, tr, instrs = traducir_con_tipos(op, l1, tipo(l1), l2, tipo(l2))
    print(f"  Expresión: {l1}({tipo(l1)}) {op} {l2}({tipo(l2)})  → tipo resultado: {tr}")
    for instr in instrs:
        print(f'    {instr}')
    print()

  Expresión: i(int) + x(float)  → tipo resultado: float
    t1 := inttofloat i
    t2 := t1 + x

  Expresión: x(float) * y(float)  → tipo resultado: float
    t3 := x * y

  Expresión: x(float) - i(int)  → tipo resultado: float
    t4 := inttofloat i
    t5 := x - t4

  Expresión: i(int) + j(int)  → tipo resultado: int
    t6 := i + j



### 🟢 E20 — Acceso a arreglo unidimensional

**Enunciado:** Genera el código de tres direcciones para acceder a `A[i]` donde `base=100`, `ancho=4`.

In [62]:
# ============================================================
# E20 — RESUELTO
# ============================================================
reset()

def acceso_arreglo_1d(nombre, base, indice, ancho):
    """Genera código para A[i]. Fórmula: dir = base + i * ancho."""
    instrucciones = []

    t1 = nueva_temporal()  # desplazamiento
    instrucciones.append(f'{t1} := {indice} * {ancho}')

    t2 = nueva_temporal()  # dirección
    instrucciones.append(f'{t2} := {base} + {t1}')

    t3 = nueva_temporal()  # valor
    instrucciones.append(f'{t3} := *{t2}    ; carga {nombre}[{indice}]')

    return t3, instrucciones

resultado, codigo = acceso_arreglo_1d('A', base=100, indice='i', ancho=4)

print("  Acceso a arreglo 1D: A[i]")
print("  base_A=100, ancho=4 bytes")
print("  Fórmula: dir(A[i]) = 100 + i * 4")
print("\n  Código de tres direcciones:")
for instr in codigo:
    print(f'    {instr}')
print(f"\n  Resultado en: {resultado}")

  Acceso a arreglo 1D: A[i]
  base_A=100, ancho=4 bytes
  Fórmula: dir(A[i]) = 100 + i * 4

  Código de tres direcciones:
    t1 := i * 4
    t2 := 100 + t1
    t3 := *t2    ; carga A[i]

  Resultado en: t3


### 🟢 E21 — Acceso a arreglo bidimensional

**Enunciado:** Genera el código de tres direcciones para `M[i][j]` con `n2=5`, `base=200`, `ancho=8`. Fórmula: `base + (i*n2 + j) * ancho`.

In [63]:
# ============================================================
# E21 — RESUELTO
# ============================================================
reset()

def acceso_arreglo_2d(nombre, base, i, j, n2, ancho):
    """
    Genera código para M[i][j].
    Fórmula: dir = base + (i * n2 + j) * ancho
    """
    instrucciones = []

    t1 = nueva_temporal()  # i * n2
    instrucciones.append(f'{t1} := {i} * {n2}')

    t2 = nueva_temporal()  # t1 + j
    instrucciones.append(f'{t2} := {t1} + {j}')

    t3 = nueva_temporal()  # t2 * ancho
    instrucciones.append(f'{t3} := {t2} * {ancho}')

    t4 = nueva_temporal()  # base + t3
    instrucciones.append(f'{t4} := {base} + {t3}')

    t5 = nueva_temporal()  # carga el valor
    instrucciones.append(f'{t5} := *{t4}    ; carga {nombre}[{i}][{j}]')

    return t5, instrucciones

resultado, codigo = acceso_arreglo_2d('M', base=200, i='i', j='j', n2=5, ancho=8)

print("  Acceso a arreglo 2D: M[i][j]")
print("  n1=4, n2=5, base=200, ancho=8 bytes")
print("  Fórmula: dir = 200 + (i*5 + j) * 8")
print("\n  Código de tres direcciones:")
for instr in codigo:
    print(f'    {instr}')
print(f"\n  Resultado en: {resultado}")

# Verificación numérica: i=2, j=3
i_val, j_val = 2, 3
dir_esperada = 200 + (i_val * 5 + j_val) * 8
print(f"\n  Verificación: M[2][3] → dirección = 200 + (2*5+3)*8 = {dir_esperada}")

  Acceso a arreglo 2D: M[i][j]
  n1=4, n2=5, base=200, ancho=8 bytes
  Fórmula: dir = 200 + (i*5 + j) * 8

  Código de tres direcciones:
    t1 := i * 5
    t2 := t1 + j
    t3 := t2 * 8
    t4 := 200 + t3
    t5 := *t4    ; carga M[i][j]

  Resultado en: t5

  Verificación: M[2][3] → dirección = 200 + (2*5+3)*8 = 304


### 🟢 E22 — Traducir expresión compleja con temporales

**Enunciado:** Traduce a código de tres direcciones `(a + b) * (c - d) / e`. Verifica con `a=6, b=2, c=10, d=4, e=3` → resultado esperado: `16.0`.

In [64]:
# ============================================================
# E22 — RESUELTO
# ============================================================
reset()

# Expresión: (a + b) * (c - d) / e
# Paso a paso:
#   t1 = a + b
#   t2 = c - d
#   t3 = t1 * t2
#   t4 = t3 / e

cuadruplos = [
    ('+', 'a',  'b',  't1'),  # t1 := a + b
    ('-', 'c',  'd',  't2'),  # t2 := c - d
    ('*', 't1', 't2', 't3'),  # t3 := t1 * t2
    ('/', 't3', 'e',  't4'),  # t4 := t3 / e
]

print("  Expresión: (a + b) * (c - d) / e")
print("\n  Cuádruplos:")
print(f"  {'#':>3} {'Op':>4} {'Arg1':>5} {'Arg2':>5} {'Res':>5}")
print("  " + "-"*28)
for i, q in enumerate(cuadruplos):
    print(f"  ({i}) {q[0]:>4} {q[1]:>5} {q[2]:>5} {q[3]:>5}")

# Ejecución
mem = {'a': 6, 'b': 2, 'c': 10, 'd': 4, 'e': 3}
ops = {'+': lambda x,y: x+y, '-': lambda x,y: x-y,
       '*': lambda x,y: x*y, '/': lambda x,y: x/y}
for op, a1, a2, res in cuadruplos:
    mem[res] = ops[op](mem[a1], mem[a2])

esperado = (6+2)*(10-4)/3
print(f"\n  Resultado: t4 = {mem['t4']}  (esperado: {esperado})")
print(f"  {'✅ Correcto' if mem['t4'] == esperado else '❌ Error'}")

  Expresión: (a + b) * (c - d) / e

  Cuádruplos:
    #   Op  Arg1  Arg2   Res
  ----------------------------
  (0)    +     a     b    t1
  (1)    -     c     d    t2
  (2)    *    t1    t2    t3
  (3)    /    t3     e    t4

  Resultado: t4 = 16.0  (esperado: 16.0)
  ✅ Correcto


### 🟢 E23 — Expresión con menos unario

**Enunciado:** Genera cuádruplos para `x = -a + b * (-c)`.

In [65]:
# ============================================================
# E23 — RESUELTO
# ============================================================
reset()

# Expresión: x = -a + b * (-c)
# Pasos:
#   t1 = -a          (uminus a)
#   t2 = -c          (uminus c)
#   t3 = b * t2      (* b t2)
#   t4 = t1 + t3     (+ t1 t3)
#   x  = t4          (:= t4)

cuadruplos = [
    ('uminus', 'a',  '',   't1'),  # t1 := -a
    ('uminus', 'c',  '',   't2'),  # t2 := -c
    ('*',      'b',  't2', 't3'),  # t3 := b * t2
    ('+',      't1', 't3', 't4'),  # t4 := t1 + t3
    (':=',     't4', '',   'x'),   # x  := t4
]

print("  Expresión: x = -a + b * (-c)")
print("\n  Cuádruplos:")
print(f"  {'#':>3} {'Op':>8} {'Arg1':>5} {'Arg2':>5} {'Res':>5}")
print("  " + "-"*36)
for i, q in enumerate(cuadruplos):
    print(f"  ({i:02d}) {q[0]:>8} {q[1]:>5} {q[2]:>5} {q[3]:>5}")

# Verificación: a=3, b=4, c=2  →  x = -3 + 4*(-2) = -3 + (-8) = -11
mem = {'a': 3, 'b': 4, 'c': 2}
for op, a1, a2, res in cuadruplos:
    if op == 'uminus':  mem[res] = -mem[a1]
    elif op == '*':     mem[res] = mem[a1] * mem[a2]
    elif op == '+':     mem[res] = mem[a1] + mem[a2]
    elif op == ':=':    mem[res] = mem[a1]
esperado = -3 + 4*(-2)
print(f"\n  Verificación (a=3,b=4,c=2): x = {mem['x']}  (esperado: {esperado})")
print(f"  {'✅ Correcto' if mem['x'] == esperado else '❌ Error'}")

  Expresión: x = -a + b * (-c)

  Cuádruplos:
    #       Op  Arg1  Arg2   Res
  ------------------------------------
  (00)   uminus     a          t1
  (01)   uminus     c          t2
  (02)        *     b    t2    t3
  (03)        +    t1    t3    t4
  (04)       :=    t4           x

  Verificación (a=3,b=4,c=2): x = -11  (esperado: -11)
  ✅ Correcto


### 🟢 E24 — Detección de subexpresiones comunes

**Enunciado:** Para `x*y + x*y`, detecta y elimina instrucciones redundantes produciendo código optimizado.

In [66]:
# ============================================================
# E24 — RESUELTO
# ============================================================
reset()

def emitir_optimizado(op, arg1, arg2, calculado, instrucciones):
    """
    Emite una instrucción evitando recalcular subexpresiones ya calculadas.
    Implementa la detección de subexpresiones comunes (CSE).
    """
    clave = (op, arg1, arg2)
    if clave in calculado:
        # La subexpresión ya fue calculada: reutilizamos el temporal
        temp = calculado[clave]
        print(f'    [CSE] {arg1} {op} {arg2} ya está en {temp} — reutilizando')
        return temp

    # Nueva subexpresión: generamos instrucción y guardamos en tabla
    temp = nueva_temporal()
    instrucciones.append(f'{temp} := {arg1} {op} {arg2}')
    calculado[clave] = temp
    return temp

calculado    = {}
instrucciones = []

print("  Expresión: x*y + x*y")
print("  Generando instrucciones con detección CSE:")
print()

t1 = emitir_optimizado('*', 'x', 'y', calculado, instrucciones)  # primera vez
t2 = emitir_optimizado('*', 'x', 'y', calculado, instrucciones)  # reutiliza
t3 = emitir_optimizado('+', t1, t2, calculado, instrucciones)

print("\n  Código optimizado generado:")
for instr in instrucciones:
    print(f'    {instr}')
print(f"\n  Resultado en: {t3}")
print(f"  Instrucciones generadas: {len(instrucciones)} (vs 3 sin optimización)")
print("\n  💡 Sin CSE: t1:=x*y, t2:=x*y (redundante), t3:=t1+t2")
print("     Con CSE: t1:=x*y, t3:=t1+t1  → ahorro de 1 instrucción")

  Expresión: x*y + x*y
  Generando instrucciones con detección CSE:

    [CSE] x * y ya está en t1 — reutilizando

  Código optimizado generado:
    t1 := x * y
    t2 := t1 + t1

  Resultado en: t2
  Instrucciones generadas: 2 (vs 3 sin optimización)

  💡 Sin CSE: t1:=x*y, t2:=x*y (redundante), t3:=t1+t2
     Con CSE: t1:=x*y, t3:=t1+t1  → ahorro de 1 instrucción


### 🟢 E25 — Conversión de notación infija a código de tres direcciones

**Enunciado:** Convierte expresiones aritméticas simples en infija a código de tres direcciones usando el algoritmo shunting-yard.

In [67]:
# ============================================================
# E25 — RESUELTO
# ============================================================

def infija_a_tres_direcciones(expresion):
    """
    Convierte una expresión infija simple (sin paréntesis) a
    código de tres direcciones usando shunting-yard.
    """
    tokens      = expresion.split()
    precedencia = {'+': 1, '-': 1, '*': 2, '/': 2}
    operadores  = set('+-*/')

    # ─── Paso 1: Infija → Postfija (Shunting-Yard) ────────────────
    salida = []
    pila   = []
    for token in tokens:
        if token in operadores:
            while (pila and pila[-1] in operadores and
                   precedencia[pila[-1]] >= precedencia[token]):
                salida.append(pila.pop())
            pila.append(token)
        else:
            salida.append(token)  # operando
    while pila:
        salida.append(pila.pop())

    # ─── Paso 2: Postfija → Instrucciones de 3 direcciones ────────
    pila_ops    = []
    instrucciones = []
    ctr = count(1)

    for token in salida:
        if token in operadores:
            op2  = pila_ops.pop()
            op1  = pila_ops.pop()
            temp = f't{next(ctr)}'
            instrucciones.append(f'{temp} := {op1} {token} {op2}')
            pila_ops.append(temp)
        else:
            pila_ops.append(token)

    resultado = pila_ops[0] if pila_ops else None
    return instrucciones, resultado

exprs = [
    'a + b * c',
    'x * y - z',
    'a + b + c',
    'a * b + c * d',
]

for expr in exprs:
    instrs, res = infija_a_tres_direcciones(expr)
    print(f"  Expresión: {expr}")
    for instr in instrs:
        print(f'    {instr}')
    print(f"  Resultado en: {res}\n")

  Expresión: a + b * c
    t1 := b * c
    t2 := a + t1
  Resultado en: t2

  Expresión: x * y - z
    t1 := x * y
    t2 := t1 - z
  Resultado en: t2

  Expresión: a + b + c
    t1 := a + b
    t2 := t1 + c
  Resultado en: t2

  Expresión: a * b + c * d
    t1 := a * b
    t2 := c * d
    t3 := t1 + t2
  Resultado en: t3



---
## 🔷 SECCIÓN E — Expresiones Booleanas y Evaluación de Cortocircuito
---

### 🟢 E26 — Traducción de expresión booleana simple a saltos

**Enunciado:** Traduce `a < b` a código de tres direcciones con saltos hacia `L_verdadero` y `L_falso`.

In [68]:
# ============================================================
# E26 — RESUELTO
# ============================================================
reset()

def traducir_relacional(id1, relop, id2, lv, lf):
    """
    Traduce id1 relop id2 a código de control de flujo.
    En vez de calcular un valor booleano, genera saltos.
    """
    return [
        f'if {id1} {relop} {id2} goto {lv}',
        f'goto {lf}'
    ]

L_true  = nueva_etiqueta()   # L1
L_false = nueva_etiqueta()   # L2

codigo = traducir_relacional('a', '<', 'b', L_true, L_false)

print("  Expresión booleana: a < b")
print(f"  Etiqueta verdadero : {L_true}")
print(f"  Etiqueta falso     : {L_false}")
print("\n  Código generado:")
for instr in codigo:
    print(f'    {instr}')
print("\n  💡 El enfoque de 'control de flujo' no materializa 0/1;")
print("     SALTA directamente a la rama correcta.")

  Expresión booleana: a < b
  Etiqueta verdadero : L1
  Etiqueta falso     : L2

  Código generado:
    if a < b goto L1
    goto L2

  💡 El enfoque de 'control de flujo' no materializa 0/1;
     SALTA directamente a la rama correcta.


### 🟢 E27 — Cortocircuito con AND

**Enunciado:** Implementa la evaluación de cortocircuito para `(a < b) AND (c > d)`.

In [69]:
# ============================================================
# E27 — RESUELTO
# ============================================================
reset()

# Expresión: (a < b) AND (c > d)
# Regla AND:
#   E1.verdadero = L_e2  (si E1 es verdad, hay que evaluar E2)
#   E1.falso     = L_false  (si E1 es falsa, toda la expr es falsa → cortocircuito)
#   E2.verdadero = L_true
#   E2.falso     = L_false

L_true  = nueva_etiqueta()   # L1
L_false = nueva_etiqueta()   # L2
L_e2    = nueva_etiqueta()   # L3: inicio de E2

codigo_and = [
    # Bloque de E1: a < b
    f'if a < b goto {L_e2}',   # verdad → evalúa E2
    f'goto {L_false}',          # falso  → cortocircuito (toda la expr es falsa)
    # Bloque de E2
    f'{L_e2}:',
    f'if c > d goto {L_true}',
    f'goto {L_false}',
]

print("  Expresión: (a < b) AND (c > d)")
print()
for instr in codigo_and:
    prefix = '  ' if instr.endswith(':') else '    '
    print(f"{prefix}{instr}")
print("\n  💡 Si a<b es FALSO → goto L_false sin evaluar c>d (cortocircuito).")

  Expresión: (a < b) AND (c > d)

    if a < b goto L3
    goto L2
  L3:
    if c > d goto L1
    goto L2

  💡 Si a<b es FALSO → goto L_false sin evaluar c>d (cortocircuito).


### 🟢 E28 — Cortocircuito con OR

**Enunciado:** Implementa la generación de código para `(x < y) OR (p > q)` con cortocircuito.

In [70]:
# ============================================================
# E28 — RESUELTO
# ============================================================
reset()

# Regla OR:
#   E1.verdadero = L_true   (si E1 es verdad, toda la expr es verdad → cortocircuito)
#   E1.falso     = L_e2     (si E1 es falsa, hay que evaluar E2)
#   E2.verdadero = L_true
#   E2.falso     = L_false

L_true  = nueva_etiqueta()   # L1
L_false = nueva_etiqueta()   # L2
L_e2    = nueva_etiqueta()   # L3: inicio de E2

codigo_or = [
    # Bloque E1: x < y
    f'if x < y goto {L_true}',   # verdad → cortocircuito (toda la expr es verdad)
    f'goto {L_e2}',               # falso  → evalúa E2
    # Bloque E2
    f'{L_e2}:',
    f'if p > q goto {L_true}',
    f'goto {L_false}',
]

print("  Expresión: (x < y) OR (p > q)")
print()
for instr in codigo_or:
    prefix = '  ' if instr.endswith(':') else '    '
    print(f"{prefix}{instr}")
print("\n  💡 Si x<y es VERDADERO → goto L_true sin evaluar p>q (cortocircuito).")

  Expresión: (x < y) OR (p > q)

    if x < y goto L1
    goto L3
  L3:
    if p > q goto L1
    goto L2

  💡 Si x<y es VERDADERO → goto L_true sin evaluar p>q (cortocircuito).


### 🟢 E29 — Traducir NOT con cortocircuito

**Enunciado:** Implementa la traducción de `NOT (a < b)`. La regla semántica del NOT intercambia las etiquetas verdadero y falso.

In [71]:
# ============================================================
# E29 — RESUELTO
# ============================================================
reset()

def traducir_not(id1, relop, id2, lv, lf):
    """
    Genera código para NOT (id1 relop id2).
    Regla semántica de NOT:
        E1.verdadero = E.falso
        E1.falso     = E.verdadero
    Es decir, simplemente se invierten las etiquetas.
    No se genera ninguna instrucción extra.
    """
    # Intercambiamos las etiquetas al llamar a traducir_relacional
    return traducir_relacional(id1, relop, id2, lf, lv)  # lf↔lv intercambiados

L_true  = nueva_etiqueta()   # L1
L_false = nueva_etiqueta()   # L2

codigo = traducir_not('a', '<', 'b', L_true, L_false)

print("  Expresión: NOT (a < b)")
print(f"  L_true={L_true}, L_false={L_false}")
print("\n  Código generado:")
for instr in codigo:
    print(f'    {instr}')
print("\n  💡 NOT (a < b) salta a L_true cuando a >= b.")
print("     El compilador NO genera instrucciones extra: solo invierte etiquetas.")

  Expresión: NOT (a < b)
  L_true=L1, L_false=L2

  Código generado:
    if a < b goto L2
    goto L1

  💡 NOT (a < b) salta a L_true cuando a >= b.
     El compilador NO genera instrucciones extra: solo invierte etiquetas.


### 🟢 E30 — Expresión booleana compuesta

**Enunciado:** Genera el código para `((a < b) AND (c > d)) OR (e == f)`. Respeta la precedencia: AND antes que OR.

In [72]:
# ============================================================
# E30 — RESUELTO
# ============================================================
reset()

# Expresión: ((a < b) AND (c > d)) OR (e == f)
#
# Estructura con etiquetas:
#   AND parte izquierda del OR:
#     E1 = (a < b): si VERDAD → evalúa E2 del AND; si FALSO → va a L_or_e2
#     E2 = (c > d): si VERDAD → toda la expr es VERDAD; si FALSO → va a L_or_e2
#   OR parte derecha:
#     E3 = (e == f): si VERDAD → L_true; si FALSO → L_false

L_true   = nueva_etiqueta()   # L1
L_false  = nueva_etiqueta()   # L2
L_and_e2 = nueva_etiqueta()   # L3: inicio de (c > d)
L_or_e2  = nueva_etiqueta()   # L4: inicio de (e == f)

codigo = [
    # ── Bloque AND: (a < b) AND (c > d) ──────────────────────────
    f'if a < b goto {L_and_e2}',  # E1 verdad → evalúa E2 del AND
    f'goto {L_or_e2}',             # E1 falsa  → AND es falso, va al OR
    f'{L_and_e2}:',
    f'if c > d goto {L_true}',    # E2 verdad → AND verdad → expr verdad
    f'goto {L_or_e2}',             # E2 falsa  → AND falso, va al OR

    # ── Bloque OR: (e == f) ───────────────────────────────────────
    f'{L_or_e2}:',
    f'if e == f goto {L_true}',
    f'goto {L_false}',
]

print("  Expresión: ((a < b) AND (c > d)) OR (e == f)")
print()
for instr in codigo:
    prefix = '  ' if instr.endswith(':') else '    '
    print(f"{prefix}{instr}")
print("\n  💡 AND tiene mayor precedencia que OR.")
print("     Si el AND falla, el OR evalúa su segunda parte (e == f).")

  Expresión: ((a < b) AND (c > d)) OR (e == f)

    if a < b goto L3
    goto L4
  L3:
    if c > d goto L1
    goto L4
  L4:
    if e == f goto L1
    goto L2

  💡 AND tiene mayor precedencia que OR.
     Si el AND falla, el OR evalúa su segunda parte (e == f).


### 🟢 E31 — Comparar evaluación estricta vs cortocircuito

**Enunciado:** Demuestra la diferencia entre evaluación estricta (evalúa ambos operandos siempre) y evaluación de cortocircuito (evaluación lazy). Cuando `e2` lanza una excepción, la estricta falla, la lazy no.

In [73]:
# ============================================================
# E31 — RESUELTO
# ============================================================

def evaluar_estricto(e1, e2, op):
    """
    Evaluación estricta: ambos operandos se evalúan ANTES de aplicar op.
    Los valores ya vienen computados como parámetros.
    """
    if op == 'AND': return e1 and e2
    if op == 'OR':  return e1 or  e2
    raise ValueError(f'Operador {op} desconocido')

def evaluar_cortocircuito(get_e1, get_e2, op):
    """
    Evaluación perezosa (lazy): solo llama a get_e2() si es necesario.
    get_e1 y get_e2 son callables (lambdas o funciones).
    """
    v1 = get_e1()
    if op == 'AND':
        if not v1: return False   # cortocircuito: e2 no se evalúa
        return bool(get_e2())
    if op == 'OR':
        if v1: return True        # cortocircuito: e2 no se evalúa
        return bool(get_e2())
    raise ValueError(f'Operador {op} desconocido')

print("  ── Test 1: AND donde e1=False, e2 lanza excepción ──")
try:
    r = evaluar_estricto(False, 1/0, 'AND')  # 1/0 se evalúa ANTES de llamar
    print(f"  Estricto       : {r}")
except ZeroDivisionError:
    print("  Estricto       : ❌ ERROR — e2 evaluado innecesariamente")

try:
    r = evaluar_cortocircuito(lambda: False, lambda: 1/0, 'AND')
    print(f"  Cortocircuito  : ✅ Resultado={r} (e2 nunca se llamó)")
except ZeroDivisionError:
    print("  Cortocircuito  : ❌ ERROR")

print("\n  ── Test 2: OR donde e1=True, e2 lanza excepción ──")
try:
    r = evaluar_estricto(True, 1/0, 'OR')
    print(f"  Estricto       : {r}")
except ZeroDivisionError:
    print("  Estricto       : ❌ ERROR")

try:
    r = evaluar_cortocircuito(lambda: True, lambda: 1/0, 'OR')
    print(f"  Cortocircuito  : ✅ Resultado={r} (e2 nunca se llamó)")
except ZeroDivisionError:
    print("  Cortocircuito  : ❌ ERROR")

print("\n  💡 El cortocircuito es ESENCIAL para expresiones como:")
print("     if (p != NULL and p->val == 10)  — sin él, falla en NULL.")

  ── Test 1: AND donde e1=False, e2 lanza excepción ──
  Estricto       : ❌ ERROR — e2 evaluado innecesariamente
  Cortocircuito  : ✅ Resultado=False (e2 nunca se llamó)

  ── Test 2: OR donde e1=True, e2 lanza excepción ──
  Estricto       : ❌ ERROR
  Cortocircuito  : ✅ Resultado=True (e2 nunca se llamó)

  💡 El cortocircuito es ESENCIAL para expresiones como:
     if (p != NULL and p->val == 10)  — sin él, falla en NULL.


### 🟢 E32 — Expresión booleana con tres operandos AND

**Enunciado:** Genera el código para `((x > 0) AND (y > 0)) AND (z > 0)` con cortocircuito.

In [74]:
# ============================================================
# E32 — RESUELTO
# ============================================================
reset()

# Expresión: ((x > 0) AND (y > 0)) AND (z > 0)
# Asocia a la izquierda: evalúa el AND interno primero.
#
# AND interno: (x > 0) AND (y > 0)
#   E1 = x > 0: verdad → evalúa E2;  falso → todo el AND es falso
#   E2 = y > 0: verdad → AND interno verdad; falso → AND interno falso
#
# AND externo: (AND_interno) AND (z > 0)
#   Si AND interno verdad → evalúa z > 0
#   Si AND interno falso  → toda la expr es falsa

L_true  = nueva_etiqueta()   # L1
L_false = nueva_etiqueta()   # L2
L_y     = nueva_etiqueta()   # L3: inicio de (y > 0)
L_z     = nueva_etiqueta()   # L4: inicio de (z > 0)

codigo = [
    # ── AND interno: (x > 0) AND (y > 0) ─────────────────────────
    f'if x > 0 goto {L_y}',    # x>0 verdad → evalúa y>0
    f'goto {L_false}',          # x>0 falso  → cortocircuito
    f'{L_y}:',
    f'if y > 0 goto {L_z}',    # y>0 verdad → evalúa z>0
    f'goto {L_false}',          # y>0 falso  → cortocircuito

    # ── AND externo: AND_interno AND (z > 0) ─────────────────────
    f'{L_z}:',
    f'if z > 0 goto {L_true}', # z>0 verdad → toda la expr verdad
    f'goto {L_false}',          # z>0 falso  → toda la expr falsa
]

print("  Expresión: ((x > 0) AND (y > 0)) AND (z > 0)")
print()
for instr in codigo:
    prefix = '  ' if instr.endswith(':') else '    '
    print(f"{prefix}{instr}")
print("\n  💡 El cortocircuito evita evaluar y y z si x ya es ≤ 0.")

  Expresión: ((x > 0) AND (y > 0)) AND (z > 0)

    if x > 0 goto L3
    goto L2
  L3:
    if y > 0 goto L4
    goto L2
  L4:
    if z > 0 goto L1
    goto L2

  💡 El cortocircuito evita evaluar y y z si x ya es ≤ 0.


### 🟢 E33 — Conteo de instrucciones con/sin cortocircuito

**Enunciado:** Para un AND de `n` términos, calcula cuántas instrucciones genera cada enfoque y compáralos.

In [75]:
# ============================================================
# E33 — RESUELTO
# ============================================================

def instrucciones_and(n, cortocircuito=True):
    """
    Estima el número de instrucciones de código de 3 direcciones
    para un AND de n términos.
    """
    if cortocircuito:
        # Con cortocircuito:
        # Cada término genera exactamente 2 instrucciones:
        #   if condicion goto L_sig  (1 instrucción)
        #   goto L_false              (1 instrucción)
        # más n-1 etiquetas (no cuentan como instrucciones ejecutables)
        return n * 2

    else:
        # Sin cortocircuito (evaluación estricta, genera valores 0/1):
        # Por cada término: 2 instrucciones (if + asignación a 0 o 1)
        # Luego n-1 instrucciones AND acumuladas
        # Más 1 salto final condicional
        instrucciones_por_termino = 2   # calcular valor booleano explícito
        combinaciones = n - 1           # t_and := t1 AND t2, etc.
        salto_final   = 1
        return n * instrucciones_por_termino + combinaciones + salto_final

print("  Instrucciones para AND de n términos:")
print(f"  {'n':>4} | {'Cortocircuito':>16} | {'Estricta':>12} | {'Ahorro':>8}")
print("  " + "-"*50)
for n in [1, 2, 3, 5, 10, 20]:
    cc = instrucciones_and(n, cortocircuito=True)
    es = instrucciones_and(n, cortocircuito=False)
    print(f"  {n:>4} | {cc:>16} | {es:>12} | {es-cc:>8}")

print("\n  💡 Cuanto más larga la cadena de ANDs, mayor el ahorro")
print("     del cortocircuito. Para n=20: 40 vs 60 instrucciones.")

  Instrucciones para AND de n términos:
     n |    Cortocircuito |     Estricta |   Ahorro
  --------------------------------------------------
     1 |                2 |            3 |        1
     2 |                4 |            6 |        2
     3 |                6 |            9 |        3
     5 |               10 |           15 |        5
    10 |               20 |           30 |       10
    20 |               40 |           60 |       20

  💡 Cuanto más larga la cadena de ANDs, mayor el ahorro
     del cortocircuito. Para n=20: 40 vs 60 instrucciones.


---
## 🔷 SECCIÓN F — Sentencias de Control y Retroceso (Backpatching)
---

### 🟢 E34 — Traducción de sentencia if-else

**Enunciado:** Genera código de tres direcciones para `if (E) S1 else S2`.

In [76]:
# ============================================================
# E34 — RESUELTO
# ============================================================
reset()

def traducir_if_else(cond_id1, cond_relop, cond_id2,
                     codigo_s1, codigo_s2):
    """
    Genera código para: if (id1 relop id2) S1 else S2

    Esqueleto:
        if id1 relop id2 goto L_true
        goto L_false
      L_true:
        [S1]
        goto L_sig
      L_false:
        [S2]
      L_sig:
    """
    L_true  = nueva_etiqueta()
    L_false = nueva_etiqueta()
    L_sig   = nueva_etiqueta()

    codigo = []
    codigo.append(f'if {cond_id1} {cond_relop} {cond_id2} goto {L_true}')
    codigo.append(f'goto {L_false}')
    codigo.append(f'{L_true}:')
    codigo.extend(codigo_s1)
    codigo.append(f'goto {L_sig}')
    codigo.append(f'{L_false}:')
    codigo.extend(codigo_s2)
    codigo.append(f'{L_sig}:')
    return codigo

s1 = ['y := x', 'x := 0']   # then: y=x; x=0;
s2 = ['y := 0']              # else: y=0;

codigo = traducir_if_else('x', '>', '0', s1, s2)

print("  Sentencia: if (x > 0) { y=x; x=0; } else { y=0; }")
print("\n  Código de tres direcciones:")
for instr in codigo:
    prefix = '  ' if instr.endswith(':') else '    '
    print(f"{prefix}{instr}")

  Sentencia: if (x > 0) { y=x; x=0; } else { y=0; }

  Código de tres direcciones:
    if x > 0 goto L1
    goto L2
  L1:
    y := x
    x := 0
    goto L3
  L2:
    y := 0
  L3:


### 🟢 E35 — Traducción de sentencia while

**Enunciado:** Genera código de tres direcciones para `while (E) S`.

In [77]:
# ============================================================
# E35 — RESUELTO
# ============================================================
reset()

def traducir_while(cond_id1, cond_relop, cond_id2, codigo_cuerpo):
    """
    Genera código para: while (id1 relop id2) { cuerpo }

    Esqueleto:
      L_inicio:
        if id1 relop id2 goto L_cuerpo
        goto L_sig
      L_cuerpo:
        [cuerpo]
        goto L_inicio
      L_sig:
    """
    L_inicio = nueva_etiqueta()
    L_cuerpo = nueva_etiqueta()
    L_sig    = nueva_etiqueta()

    codigo = []
    codigo.append(f'{L_inicio}:')
    codigo.append(f'if {cond_id1} {cond_relop} {cond_id2} goto {L_cuerpo}')
    codigo.append(f'goto {L_sig}')
    codigo.append(f'{L_cuerpo}:')
    codigo.extend(codigo_cuerpo)
    codigo.append(f'goto {L_inicio}')
    codigo.append(f'{L_sig}:')
    return codigo

cuerpo = ['sum := sum + i', 'i := i + 1']
codigo = traducir_while('i', '<', 'n', cuerpo)

print("  Sentencia: while (i < n) { sum=sum+i; i=i+1; }")
print("\n  Código de tres direcciones:")
for instr in codigo:
    prefix = '  ' if instr.endswith(':') else '    '
    print(f"{prefix}{instr}")

  Sentencia: while (i < n) { sum=sum+i; i=i+1; }

  Código de tres direcciones:
  L1:
    if i < n goto L2
    goto L3
  L2:
    sum := sum + i
    i := i + 1
    goto L1
  L3:


### 🟢 E36 — Técnica de Retroceso (Backpatching) — fundamentos

**Enunciado:** Implementa `crear_lista`, `fusionar` y `retroceder`. Demuestra su uso para rellenar saltos pendientes en una sola pasada.

In [78]:
# ============================================================
# E36 — RESUELTO
# ============================================================

codigo_bp = []   # arreglo de instrucciones con posibles 'agujeros'

def emitir_bp(tipo, arg1=None, arg2=None, destino=None):
    """Emite una instrucción y retorna su índice."""
    idx = len(codigo_bp)
    codigo_bp.append({'tipo': tipo, 'arg1': arg1, 'arg2': arg2, 'dest': destino})
    return idx

def crear_lista(i):
    """Crea una lista de retroceso con el índice i."""
    return [i]

def fusionar(p1, p2):
    """Concatena dos listas de retroceso."""
    return p1 + p2

def retroceder(p, i):
    """Rellena el destino de todos los saltos en la lista p con la dirección i."""
    for idx in p:
        codigo_bp[idx]['dest'] = i
        print(f'    [retroceso] instrucción {idx} → dest := {i}')

def mostrar_codigo_bp():
    for i, instr in enumerate(codigo_bp):
        dest = instr['dest'] if instr['dest'] is not None else '???'
        tipo = instr['tipo']
        if tipo == 'goto':
            print(f"    ({i:02d})  goto {dest}")
        elif tipo == 'if':
            print(f"    ({i:02d})  if {instr['arg1']} goto {dest}")
        else:
            print(f"    ({i:02d})  {tipo} {instr.get('arg1','')} := {instr.get('arg2','')}")

print("  === Simulación de Backpatching para: if (a < b) S1 ; S2 ===")
print()

# Emitir instrucciones con destinos pendientes
idx0 = emitir_bp('if',   arg1='a < b')   # if a<b goto ???
idx1 = emitir_bp('goto')                  # goto ??? (si falso)

print("  Código con agujeros:")
mostrar_codigo_bp()

# Cuerpo de S1
idx2 = emitir_bp('asign', arg1='x', arg2='1')   # x := 1
idx3 = emitir_bp('goto')                          # goto L_sig (después de S1)

# Ahora conocemos L_true = idx2 → retrocede el 'if'
lista_verd = crear_lista(idx0)
lista_fals = crear_lista(idx1)

print("\n  Retrocediendo destinos:")
retroceder(lista_verd, idx2)   # El if salta al cuerpo S1

# S2 empieza aquí
idx4 = emitir_bp('asign', arg1='x', arg2='0')
idx5 = len(codigo_bp)  # posición de la siguiente instrucción = L_sig

retroceder(lista_fals, idx4)   # El goto-falso salta a S2
retroceder(crear_lista(idx3), idx5)  # El goto-post-S1 salta a L_sig

print("\n  Código final (agujeros rellenados):")
mostrar_codigo_bp()

  === Simulación de Backpatching para: if (a < b) S1 ; S2 ===

  Código con agujeros:
    (00)  if a < b goto ???
    (01)  goto ???

  Retrocediendo destinos:
    [retroceso] instrucción 0 → dest := 2
    [retroceso] instrucción 1 → dest := 4
    [retroceso] instrucción 3 → dest := 5

  Código final (agujeros rellenados):
    (00)  if a < b goto 2
    (01)  goto 4
    (02)  asign x := 1
    (03)  goto 5
    (04)  asign x := 0


### 🟢 E37 — Backpatching para sentencia if sin else

**Enunciado:** Usa backpatching para generar código para `if (x != y) { z = x + y; }` en una sola pasada.

In [79]:
# ============================================================
# E37 — RESUELTO
# ============================================================

codigo_bp = []   # reset del arreglo

print("  Backpatching para: if (x != y) { z = x + y; }")
print("  (una sola pasada, sin conocer las etiquetas de destino por adelantado)")
print()

# ── 1. Emitir la condición con destino pendiente ───────────────────
idx_if   = emitir_bp('if',   arg1='x != y')  # if x != y goto ???  (L_true)
idx_goto = emitir_bp('goto')                  # goto ???            (L_sig)

lista_verdadero = crear_lista(idx_if)    # lista de saltos que van al cuerpo
lista_falso     = crear_lista(idx_goto)  # lista de saltos que salen del if

# ── 2. Inicio del cuerpo — ahora conocemos L_true ─────────────────
L_true = len(codigo_bp)   # posición actual = donde empieza el cuerpo
print(f"  Retrocediendo lista_verdadero → L_true = instrucción {L_true}:")
retroceder(lista_verdadero, L_true)

# ── 3. Emitir el cuerpo ───────────────────────────────────────────
idx_t1 = emitir_bp('asign', arg1='t1', arg2='x + y')
idx_z  = emitir_bp('asign', arg1='z',  arg2='t1')

# ── 4. Siguiente instrucción = L_sig ──────────────────────────────
L_sig = len(codigo_bp)   # posición de la instrucción que sigue al if
print(f"  Retrocediendo lista_falso     → L_sig  = instrucción {L_sig}:")
retroceder(lista_falso, L_sig)

print("\n  Código final:")
mostrar_codigo_bp()

  Backpatching para: if (x != y) { z = x + y; }
  (una sola pasada, sin conocer las etiquetas de destino por adelantado)

  Retrocediendo lista_verdadero → L_true = instrucción 2:
    [retroceso] instrucción 0 → dest := 2
  Retrocediendo lista_falso     → L_sig  = instrucción 4:
    [retroceso] instrucción 1 → dest := 4

  Código final:
    (00)  if x != y goto 2
    (01)  goto 4
    (02)  asign t1 := x + y
    (03)  asign z := t1


### 🟢 E38 — Backpatching para sentencia while

**Enunciado:** Aplica backpatching para `while (i < 10) { i = i + 1; }` en una sola pasada.

In [80]:
# ============================================================
# E38 — RESUELTO
# ============================================================

codigo_bp = []

print("  Backpatching para: while (i < 10) { i = i + 1; }")
print()

# ── M1: posición de inicio del bucle (para el goto de vuelta) ─────
M1 = len(codigo_bp)
print(f"  M1 (inicio del bucle) = instrucción {M1}")

# ── Condición con destino pendiente ───────────────────────────────
idx_if   = emitir_bp('if',   arg1='i < 10')  # if i < 10 goto ???  (cuerpo)
idx_goto = emitir_bp('goto')                  # goto ???            (salida)

lista_verdadero = crear_lista(idx_if)    # saltos al cuerpo
lista_falso     = crear_lista(idx_goto)  # saltos a la salida

# ── M2: inicio del cuerpo → retrocede lista_verdadero ─────────────
M2 = len(codigo_bp)
print(f"  M2 (inicio del cuerpo) = instrucción {M2}")
print(f"  Retrocediendo lista_verdadero → {M2}:")
retroceder(lista_verdadero, M2)

# ── Cuerpo del bucle ──────────────────────────────────────────────
emitir_bp('asign', arg1='i', arg2='i + 1')

# ── goto M1 (salto de vuelta al inicio) ───────────────────────────
emitir_bp('goto', destino=M1)  # este destino ya se conoce

# ── L_sig: posición de salida del bucle ───────────────────────────
L_sig = len(codigo_bp)
print(f"  L_sig (salida del bucle) = instrucción {L_sig}")
print(f"  Retrocediendo lista_falso → {L_sig}:")
retroceder(lista_falso, L_sig)

print("\n  Código final:")
mostrar_codigo_bp()

  Backpatching para: while (i < 10) { i = i + 1; }

  M1 (inicio del bucle) = instrucción 0
  M2 (inicio del cuerpo) = instrucción 2
  Retrocediendo lista_verdadero → 2:
    [retroceso] instrucción 0 → dest := 2
  L_sig (salida del bucle) = instrucción 4
  Retrocediendo lista_falso → 4:
    [retroceso] instrucción 1 → dest := 4

  Código final:
    (00)  if i < 10 goto 2
    (01)  goto 4
    (02)  asign i := i + 1
    (03)  goto 0


### 🟢 E39 — Traducir bucle for a código de tres direcciones

**Enunciado:** Implementa `traducir_for` usando la equivalencia: `for(init; cond; inc) { S }` ≡ `init; while(cond) { S; inc }`.

In [81]:
# ============================================================
# E39 — RESUELTO
# ============================================================
reset()

def traducir_for(var_init, val_init, cond_id1, cond_relop, cond_id2,
                 var_inc, cuerpo):
    """
    Genera código para: for (var_init = val_init; cond; var_inc++)

    Equivalente a:
        var_init := val_init
        while (cond) {
            cuerpo
            var_inc := var_inc + 1
        }
    """
    codigo = []

    # Inicialización
    codigo.append(f'{var_init} := {val_init}')

    # Cuerpo extendido: cuerpo original + incremento
    cuerpo_extendido = list(cuerpo) + [f'{var_inc} := {var_inc} + 1']

    # Generar el while equivalente
    codigo_while = traducir_while(cond_id1, cond_relop, cond_id2, cuerpo_extendido)
    codigo.extend(codigo_while)

    return codigo

# for (i = 0; i < n; i++) { sum = sum + i; }
codigo = traducir_for(
    var_init='i',  val_init='0',
    cond_id1='i',  cond_relop='<',  cond_id2='n',
    var_inc='i',
    cuerpo=['sum := sum + i']
)

print("  Sentencia: for (i = 0; i < n; i++) { sum = sum + i; }")
print("  Equivale a: i:=0; while(i<n){ sum:=sum+i; i:=i+1 }")
print("\n  Código de tres direcciones:")
for instr in codigo:
    prefix = '  ' if instr.endswith(':') else '    '
    print(f"{prefix}{instr}")

  Sentencia: for (i = 0; i < n; i++) { sum = sum + i; }
  Equivale a: i:=0; while(i<n){ sum:=sum+i; i:=i+1 }

  Código de tres direcciones:
    i := 0
  L1:
    if i < n goto L2
    goto L3
  L2:
    sum := sum + i
    i := i + 1
    goto L1
  L3:


### 🟢 E40 — Generación completa: mini-compilador de expresiones

**Enunciado:** Integra todo lo aprendido. Compila un programa con inicializaciones y un bucle `while`, genera el código de tres direcciones completo y verifica con un intérprete que `sum = 0+1+…+9 = 45`.

In [85]:
# ============================================================
# E40 — RESUELTO: Mini-compilador integrador (CORREGIDO)
# ============================================================

programa = {
    'vars': ['i', 'sum', 'n'],
    'init': [('i', '0'), ('sum', '0'), ('n', '10')],
    'while_cond': ('i', '<', 'n'),
    'while_body': [('sum', 'sum + i'), ('i', 'i + 1')]
}

# ── Generador de etiquetas únicas ─────────────────────────────────
_label_counter = [0]

def nueva_etiqueta():
    _label_counter[0] += 1
    return f'L{_label_counter[0]}'

def reset():
    _label_counter[0] = 0

def traducir_while(c1, relop, c2, cuerpo):
    """
    Genera código de tres direcciones para:
        while c1 relop c2:
            cuerpo
    """
    L_inicio = nueva_etiqueta()   # Volver aquí cada iteración
    L_cuerpo = nueva_etiqueta()   # Entrar si condición es verdadera
    L_fin    = nueva_etiqueta()   # Salir del bucle

    codigo = []
    codigo.append(f'{L_inicio}:')
    codigo.append(f'if {c1} {relop} {c2} goto {L_cuerpo}')
    codigo.append(f'goto {L_fin}')
    codigo.append(f'{L_cuerpo}:')
    for inst in cuerpo:
        codigo.append(inst)
    codigo.append(f'goto {L_inicio}')
    codigo.append(f'{L_fin}:')
    return codigo

def compilar_programa(prog):
    """Genera código de tres direcciones para el programa estructurado."""
    codigo = []

    # 1. Inicializaciones
    for var, val in prog['init']:
        codigo.append(f'{var} := {val}')

    # 2. Cuerpo del while como lista de instrucciones
    cuerpo = [f'{dest} := {expr}' for dest, expr in prog['while_body']]

    # 3. Generar el while
    c1, relop, c2 = prog['while_cond']
    codigo_while = traducir_while(c1, relop, c2, cuerpo)
    codigo.extend(codigo_while)

    return codigo


def interpretar(codigo_lista):
    """
    Intérprete básico de código de tres direcciones.
    Soporta: asignaciones (:=), goto, if var relop var goto.
    """
    mem = {}

    # Primer paso: recolectar etiquetas {nombre: índice}
    etiquetas = {}
    for i, instr in enumerate(codigo_lista):
        s = instr.strip()
        if s.endswith(':'):
            etiquetas[s.rstrip(':')] = i

    def eval_expr(expr):
        """
        Evalúa expresión simple:
          - un solo token  → número o variable
          - tres tokens    → 'a op b'
        ✅ FIX: usar if/elif en lugar de dict para evitar
           ZeroDivisionError al construir todas las ramas.
        """
        tokens = expr.strip().split()

        if len(tokens) == 1:
            v = tokens[0]
            try:
                return int(v)
            except ValueError:
                return mem.get(v, 0)

        if len(tokens) == 3:
            l, op, r = tokens

            # Resolver operandos
            try:
                lv = int(l)
            except ValueError:
                lv = mem.get(l, 0)

            try:
                rv = int(r)
            except ValueError:
                rv = mem.get(r, 0)

            # ✅ FIX: evaluar solo la operación necesaria
            if op == '+':
                return lv + rv
            elif op == '-':
                return lv - rv
            elif op == '*':
                return lv * rv
            elif op == '/':
                if rv == 0:
                    raise ZeroDivisionError(f'División por cero: {lv} / {rv}')
                return lv // rv
            else:
                print(f'  ⚠️  Operador desconocido: {op}')
                return 0

        return 0

    # Segundo paso: ejecutar instrucción por instrucción
    pc = 0
    iteraciones = 0
    MAX_ITER = 10_000

    while pc < len(codigo_lista):
        iteraciones += 1
        if iteraciones > MAX_ITER:
            print('  ⚠️  Límite de iteraciones alcanzado')
            break

        instr = codigo_lista[pc].strip()

        # Etiqueta — solo marcar posición, no hacer nada
        if instr.endswith(':'):
            pc += 1
            continue

        # Salto incondicional
        if instr.startswith('goto '):
            dest = instr.split()[1]
            pc = etiquetas[dest] + 1
            continue

        # Salto condicional: if v1 relop v2 goto Lx
        if instr.startswith('if '):
            partes = instr.split()
            # partes: ['if', v1, relop, v2, 'goto', label]
            v1  = mem.get(partes[1], 0)
            rop = partes[2]
            try:
                v2 = int(partes[3])
            except ValueError:
                v2 = mem.get(partes[3], 0)
            lbl = partes[5]

            cond = False
            if rop == '<':  cond = v1 <  v2
            elif rop == '>':  cond = v1 >  v2
            elif rop == '<=': cond = v1 <= v2
            elif rop == '>=': cond = v1 >= v2
            elif rop == '==': cond = v1 == v2
            elif rop == '!=': cond = v1 != v2

            pc = etiquetas[lbl] + 1 if cond else pc + 1
            continue

        # Asignación: lhs := rhs
        if ':=' in instr:
            lhs, rhs = instr.split(':=', 1)
            mem[lhs.strip()] = eval_expr(rhs.strip())
            pc += 1
            continue

        # Instrucción desconocida — ignorar
        pc += 1

    return mem


# ── Compilar ───────────────────────────────────────────────────────
reset()
codigo = compilar_programa(programa)

print("Código de tres direcciones compilado:")
print("-" * 40)
for i, instr in enumerate(codigo):
    prefix = '  ' if instr.strip().endswith(':') else '    '
    print(f"  ({i:02d}){prefix}{instr}")

# ── Ejecutar ───────────────────────────────────────────────────────
print()
mem = interpretar(codigo)

print("Resultado tras la ejecución:")
print("-" * 40)
for var in ['i', 'sum', 'n']:
    print(f"  {var} = {mem.get(var, '?')}")

esperado = sum(range(10))   # 0+1+2+...+9 = 45
obtenido = mem.get('sum', -1)
print(f"\n  sum esperada : {esperado}")
print(f"  sum obtenida : {obtenido}")
print(f"  {'✅ Correcto' if obtenido == esperado else '❌ Error'}")

Código de tres direcciones compilado:
----------------------------------------
  (00)    i := 0
  (01)    sum := 0
  (02)    n := 10
  (03)  L1:
  (04)    if i < n goto L2
  (05)    goto L3
  (06)  L2:
  (07)    sum := sum + i
  (08)    i := i + 1
  (09)    goto L1
  (10)  L3:

Resultado tras la ejecución:
----------------------------------------
  i = 10
  sum = 45
  n = 10

  sum esperada : 45
  sum obtenida : 45
  ✅ Correcto


---

## 📚 Resumen de Conceptos Clave

| Concepto | Descripción | Referencia (Aho et al.) |
|---|---|---|
| **Representación Intermedia (RI)** | Puente entre front-end y back-end | Cap. 6 |
| **AST** | Árbol de Sintaxis Abstracta | §6.2.1 |
| **GAD** | Grafo Acíclico Dirigido, comparte subexpresiones comunes | §6.1.1 |
| **Código de 3 direcciones** | Instrucciones con máximo 3 operandos | §6.2.1 |
| **Cuádruplos** | `(op, arg1, arg2, resultado)` — fácil de reordenar | §6.2.2 |
| **Triplos** | `(op, arg1, arg2)` — compactos, referencias posicionales | §6.2.3 |
| **Triplos indirectos** | Triplos + arreglo de punteros reordenable | §6.2.3 |
| **Esquema S-atribuido** | Traducción basada en atributos sintetizados | §6.3 |
| **Cortocircuito** | Evalúa solo los operandos necesarios en AND/OR | §6.6.2 |
| **Backpatching** | Rellena destinos de saltos en una sola pasada | §6.7 |

---

## 📖 Referencias

- **Aho, A. V., Sethi, R., & Lam, M. S.** (2011). *Compiladores: Principios, Técnicas y Herramientas* (2ª ed.). Pearson Educación de México.  
  [[PDF en línea]](https://isergiobernalesgarcia.edu.pe/wp-content/uploads/2025/10/Compiladores-Alfred-V.-Aho-Monica-S.-Lam-Ravi-Sethi-Jeffrey-D.-Ullman.pdf)

- **Moreno, M. A. et al.** (2006). *Compiladores e Intérpretes: Teoría y Práctica*. Pearson Educación.  
  [[PDF en línea]](https://profesorezequielruizgarcia.wordpress.com/wp-content/uploads/2017/06/compiladores-e-interpretes-teoria-y-practica.pdf)